# 03 Raw parse: the four tables, and mapping by meaning
`SOI` column headers are **XBRL labels**, and the report adds columns dynamically. Map them to canonical fields through the `TAG` table, by the tag they label, and read only the columns you need.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

In [ ]:
sub = load.load_sub(RDIR)
print(f"SUB: {len(sub)} current filings (superseded originals with prevrpt=1 are dropped)")
print(sub["form"].value_counts().to_string())

In [ ]:
tag = load.load_tag(RDIR)
std = tag[(tag["custom"] == "0") & tag["tag"].isin(mapping.CANONICAL.values())][["tag", "tlabel"]]
print("Canonical tags and their labels in TAG:")
print(std.to_string(index=False))

In [ ]:
header = load.soi_header(RDIR)
meta = [c for c in header if c.islower()]
axes = [c for c in header if c.endswith("Axis")]
vals = [c for c in header if c not in meta + axes]
print("metadata:", meta); print("axes:", axes); print("values:", vals)

In [ ]:
colmap = mapping.map_soi_columns(header, tag)
for field, cols in colmap.items():
    print(f"  {field:15s} <- {cols}")
print("Unmapped canonical fields:", set(mapping.CANONICAL) - set(colmap) or "none")
use = mapping.soi_usecols(header, colmap)
print(f"Reading {len(use)} of {len(header)} SOI columns; unmapped value columns to review: {[c for c in vals if c not in use]}")

In [ ]:
num = load.load_num(RDIR, tags=[reconcile.FV_TAG])        # chunked read, filtered by tag
print(f"NUM fair-value facts: {len(num):,}; dimension counts:")
print(num["dimn"].value_counts().sort_index().to_string())

**Scale.** A full release's `num.tsv` and `soi.tsv` can each be hundreds of megabytes. Read `NUM` in chunks and filter early, and read only the `SOI` columns you use, as these helpers do. DuckDB can also query the TSV files directly; convert to Parquet once if you query repeatedly.

**Your turn.** Complete `docs/schema.md`: the tag and `SOI` column behind every canonical field, and a decision on every unmapped value column (Case Study 2B, Part 1).